# N05 · Light from a Star: Flux, Distance and Photon Counting
## How much of the Sun's light would a telescope collect if the Sun sat 1 kpc away?

---

## Learning objectives

After this tutorial you will be able to:

1. State and justify the **basic physical properties** of the Sun (mass, luminosity, radius, effective temperature) and plot its optical/near-infrared **blackbody spectrum**.
2. Explain what the **parsec** is geometrically (via stellar parallax) and what it means to place a star **1 kpc** away.
3. Build the **rest-frame emitted spectrum** (spectral luminosity $L_\lambda$) of the Sun and check that it integrates back to the bolometric luminosity $L_\odot$.
4. Propagate the light to Earth with the **inverse-square law**, first in an **ideal** (vacuum, perfect telescope) case and then in a **fully realistic** case that folds in interstellar dust extinction, the atmosphere, optical throughput and detector quantum efficiency.
5. Convert a spectral flux into a **photon rate** and compute, in a narrow band over one hour, the number of photons **emitted** by the star and **received/detected** by a 10 m telescope.
6. Derive and visualise the **scaling relations** linking emitted luminosity, distance and received light, and explore how the detected signal changes when you vary the **distance** and the **telescope diameter**.

**Mathematics used:** algebra, powers and logarithms, a single definite integral (done numerically), and the geometry of a sphere.
**Physics used:** thermal (blackbody) radiation, conservation of energy / flux, geometric optics.

**Estimated time:** 3–4 hours.

---

> **How to read this notebook.** Each physical question is answered in two passes. We **first** do the clean *ideal* calculation (vacuum, perfect instrument) so the core physics is visible, and **then** we redo it in the *full-realism* case, adding the effects that a real observation must contend with. Compare the two numbers at every step — the gap between them *is* the difference between textbook physics and a real observatory.

---

## 1. Setup, constants and solar reference values

We work in SI units and let `astropy.units` carry the units for us, so every result is dimensionally checked automatically. The fundamental constants are the **NIST CODATA 2022** values ($h$, $c$, $k_B$ are *exact* in the SI since the 2019 redefinition). The solar values are the **IAU 2015 nominal** values.

| Symbol | Quantity | Value |
|--------|----------|-------|
| $h$ | Planck constant | $6.62607015\times10^{-34}\,\mathrm{J\,s}$ |
| $c$ | speed of light | $2.99792458\times10^{8}\,\mathrm{m\,s^{-1}}$ |
| $k_B$ | Boltzmann constant | $1.380649\times10^{-23}\,\mathrm{J\,K^{-1}}$ |
| $\sigma$ | Stefan–Boltzmann constant | $5.6704\times10^{-8}\,\mathrm{W\,m^{-2}\,K^{-4}}$ |
| $L_\odot$ | solar luminosity | $3.828\times10^{26}\,\mathrm{W}$ |
| $R_\odot$ | solar radius | $6.957\times10^{8}\,\mathrm{m}$ |
| $M_\odot$ | solar mass | $1.988\times10^{30}\,\mathrm{kg}$ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.constants as const

plt.rcParams.update({'figure.figsize': (8, 5), 'font.size': 12,
                     'axes.grid': True, 'grid.alpha': 0.3})

# --- Fundamental constants (astropy already knows these) ---
h       = const.h
c       = const.c
k_B     = const.k_B
sigma_sb = const.sigma_sb
b_wien  = const.b_wien

# --- Solar reference values: fill these in ---
# L_sun = ...        # use const.L_sun
# R_sun = ...        # use const.R_sun
# M_sun = ...        # use const.M_sun
# T_sun = ...        # IAU nominal effective temperature, 5772 K

# --- Distances ---
# AU  = ...          # const.au
# pc  = ...          # const.pc
# kpc = ...          # 1000 * pc

# --- YOUR CODE HERE: print the constants to check their values ---

---

## 2. Question 1.a — The basic properties of a Sun-like star

A *Sun-like star* (spectral type G2 V, a main-sequence dwarf) is characterised by four numbers:

| Property | Symbol | Value | What it controls |
|----------|--------|-------|------------------|
| Mass | $M_\odot$ | $1.99\times10^{30}$ kg | gravity, lifetime, fusion rate |
| Radius | $R_\odot$ | $6.96\times10^{8}$ m | emitting surface area |
| Effective temperature | $T_{\rm eff}$ | $5772$ K | colour and peak wavelength |
| Luminosity | $L_\odot$ | $3.83\times10^{26}$ W | total radiated power |

These four are **not independent**. To a good first approximation the Sun radiates like a **blackbody** — a perfect thermal emitter — so its surface brightness follows the **Planck law**, and its total output follows the **Stefan–Boltzmann law**

$$ L = 4\pi R^2\,\sigma\,T_{\rm eff}^4 . $$

The factor $4\pi R^2$ is the surface area of the star (a sphere) and $\sigma T^4$ is the power radiated per unit area. We will (i) define the Planck function, (ii) plot the Sun's spectrum, (iii) find where it peaks (**Wien's law**), and (iv) verify that Stefan–Boltzmann reproduces $L_\odot$.

### 2.1 The Planck function

The **spectral radiance** of a blackbody — the power emitted per unit area, per unit wavelength, per unit solid angle — is

$$ B_\lambda(T) = \frac{2hc^2}{\lambda^5}\,\frac{1}{\exp\!\left(\dfrac{hc}{\lambda k_B T}\right)-1}
\qquad [\,\mathrm{W\,m^{-2}\,nm^{-1}\,sr^{-1}}\,]. $$

The only subtlety in code is that the exponent $hc/(\lambda k_B T)$ becomes huge at short wavelengths, which would overflow `exp`; we clip it at 700 (since $e^{700}$ is already astronomically large) to stay numerically safe.

In [ ]:
def B_lambda(lam, T):
    """Planck spectral radiance per unit wavelength [W m^-2 nm^-1 sr^-1]."""
    lam = lam.to(u.m)
    # --- YOUR CODE HERE ---
    # 1. compute the dimensionless exponent hc/(lam k_B T) and clip it at 700
    # 2. return 2 h c^2 / lam^5 / (exp(exponent) - 1), with a /u.sr attached
    raise NotImplementedError

# print(B_lambda(500*u.nm, T_sun))

### 2.2 The solar spectrum in the optical / near-infrared

We now plot $B_\lambda(T_\odot)$ across $300$–$2500$ nm, i.e. from the near-ultraviolet, through the **visible** band (380–700 nm, shaded), into the **near-infrared (NIR)**.

**Wien's displacement law** tells us where the spectrum peaks:

$$ \lambda_{\rm peak} = \frac{b}{T}, \qquad b = 2.898\times10^{-3}\,\mathrm{m\,K}. $$

For $T_\odot = 5772$ K this gives $\lambda_{\rm peak}\approx 502$ nm — green light, right in the middle of the visible band and, not coincidentally, where the human eye is most sensitive.

*(A real solar spectrum is not a perfect blackbody: it is crossed by thousands of dark **absorption lines** — Fraunhofer lines: the Balmer hydrogen lines, the Ca H&K doublet, the Na D doublet, the Mg b triplet — where atoms in the cooler outer layers absorb specific wavelengths. The smooth blackbody is the **continuum** those lines are carved out of.)*

In [ ]:
# --- YOUR CODE HERE ---
# 1. build a wavelength grid 300 -> 2500 nm and evaluate B_lambda(lam, T_sun)
# 2. compute the Wien peak  lambda_peak = b_wien / T_sun
# 3. plot the spectrum, shade the visible band (380-700 nm), mark the peak


### 2.3 Closing the loop: Stefan–Boltzmann

If the Sun is a blackbody, then *integrating* the Planck law over all wavelengths and over the full sphere must give back its luminosity. Stefan–Boltzmann packages that integral into a single formula:

$$ L = 4\pi R_\odot^2\,\sigma\,T_{\rm eff}^4 . $$

Let us evaluate it and compare to the catalogued $L_\odot$. (They agree to better than 1 %; the small residual is because the IAU defines $T_{\rm eff}=5772$ K *from* the nominal $L_\odot$ and $R_\odot$, with rounding.)

In [ ]:
# --- YOUR CODE HERE ---
# Evaluate L = 4 pi R_sun^2 sigma T_sun^4 and compare to L_sun.


---

## 3. Question 1.b — Put the Sun at 1 kpc, and write down the emitted spectrum

### 3.1 What is a parsec, and what does "1 kpc away" mean?

Astronomers measure distance with **parallax**: as the Earth orbits the Sun, a nearby star appears to shift back and forth against the far background. Over six months the Earth moves by $2\,\mathrm{AU}$ (the diameter of its orbit), and the star's apparent half-shift is the **parallax angle** $p$.

The **parsec** is *defined* as the distance at which a star shows a parallax of **1 arcsecond** ($1'' = 1/3600$ of a degree) for a baseline of **1 AU**:

$$ d\,[\mathrm{pc}] = \frac{1}{p\,['']}, \qquad 1\,\mathrm{pc} = \frac{1\,\mathrm{AU}}{1''\;(\text{in radians})} = 3.086\times10^{16}\,\mathrm{m} \approx 3.26\ \text{light-years}. $$

A **kiloparsec** is $1\,\mathrm{kpc} = 1000\,\mathrm{pc} \approx 3.086\times10^{19}$ m. For scale: the Sun's nearest neighbour (Proxima Centauri) is $1.3$ pc away, and the Milky Way's disk is $\sim 30$ kpc across — so a star at **1 kpc** is a *typical, relatively nearby star within our own Galaxy*, well inside the Milky Way.

**Important consequence:** 1 kpc is a *galactic* distance, not a *cosmological* one. The recession velocity from cosmic expansion at 1 kpc is $\sim 0.07\,\mathrm{km/s}$ — utterly negligible. So the **redshift is essentially zero**: the wavelengths we receive are the same as those emitted. The "rest-frame emitted spectrum" and the "observed-frame spectrum" live on the *same wavelength grid*; only the **brightness** changes with distance.

In [ ]:
# --- YOUR CODE HERE ---
# 1. recover 1 pc from its definition: 1 AU divided by (1 arcsec expressed in radians)
# 2. estimate the Hubble-flow velocity at 1 kpc (H0 ~ 70 km/s/Mpc) and the redshift z=v/c


### 3.2 The rest-frame emitted spectrum: spectral luminosity $L_\lambda$

The Planck function $B_\lambda$ is a *surface brightness* (per unit area, per steradian). To get the **total power the star emits per unit wavelength** — its **spectral luminosity** $L_\lambda$ — we integrate over the stellar surface and over the outward hemisphere of directions.

- Integrating the radiance over the emitting hemisphere converts radiance to **surface flux**: $F_\lambda^{\rm surf} = \pi\,B_\lambda$. (The factor $\pi$, with units of steradian, is $\int\cos\theta\,d\Omega$ over a hemisphere.)
- Multiplying by the surface area $4\pi R_\odot^2$ gives the spectral luminosity:

$$ \boxed{\,L_\lambda = 4\pi R_\odot^2\cdot \pi B_\lambda(T) = 4\pi^2 R_\odot^2\,B_\lambda(T)\,}\qquad [\,\mathrm{W\,nm^{-1}}\,]. $$

This is the *rest-frame emitted spectrum* the question asks for: energy per second per nanometre, leaving the star in all directions. As a self-consistency check, $\int_0^\infty L_\lambda\,d\lambda$ must equal the bolometric luminosity $L_\odot$.

In [ ]:
def L_lambda(lam, T=T_sun, R=R_sun):
    """Spectral luminosity of a blackbody star [W / nm]."""
    # --- YOUR CODE HERE ---
    # L_lambda = 4 pi R^2 * (pi * B_lambda)
    raise NotImplementedError

# --- YOUR CODE HERE ---
# Integrate L_lambda over a wide wavelength grid (use np.trapezoid) and check
# that it returns L_sun; then plot L_lambda over the optical/NIR.


---

## 4. Question 1.c — Propagate the light to Earth and observe it with a 10 m telescope

The star radiates $L_\lambda$ in **all directions**. By the time the light reaches Earth, that power has spread over an enormous sphere. How much lands on our telescope?

We answer this **twice**: first the **ideal** case (empty space, perfect telescope), then the **fully realistic** case.

### 4.1 STEP 1 — Ideal case: the inverse-square law and a perfect aperture

Energy is conserved, so the power $L_\lambda\,d\lambda$ emitted in a wavelength bin must, at distance $d$, be spread uniformly over a sphere of area $4\pi d^2$. The power arriving **per unit area** — the **spectral flux density** — is therefore

$$ f_\lambda = \frac{L_\lambda}{4\pi d^2} \qquad [\,\mathrm{W\,m^{-2}\,nm^{-1}}\,]. $$

This $1/d^2$ falloff is the **inverse-square law**: double the distance, quarter the flux. A telescope of diameter $D$ presents a **collecting area**

$$ A_{\rm tel} = \pi\left(\frac{D}{2}\right)^2, $$

and (ideally) intercepts the power $f_\lambda\,A_{\rm tel}\,d\lambda$. A bigger mirror is quite literally a bigger bucket for catching photons.

*(Aside — resolution. A telescope is not only a light bucket; diffraction sets the finest angle it can resolve, $\theta_{\rm min}\approx 1.22\,\lambda/D$. For $D=10$ m at $\lambda=550$ nm this is $\sim0.014''$ — but Earth's atmosphere blurs ground-based images to $\sim1''$ unless adaptive optics is used. Resolution does not change the *total* number of photons collected, which is what we track here.)*

In [ ]:
# --- YOUR CODE HERE ---
# 1. set d = 1 kpc, D_tel = 10 m, and compute the collecting area A = pi (D/2)^2
# 2. ideal spectral flux at Earth: f_lambda = L_lambda / (4 pi d^2)
# 3. (optional) diffraction limit theta = 1.22 lambda / D at 550 nm
# 4. plot f_lambda vs wavelength


### 4.2 STEP 2 — Full realism: dust, atmosphere, optics and detector

Between the star and our recorded image, the light is attenuated by four effects, each a multiplicative factor between 0 and 1:

1. **Interstellar extinction (dust).** The Milky Way's disk is filled with dust grains that absorb and scatter starlight, preferentially in the blue (this also *reddens* the star). The dimming in magnitudes is $A_\lambda$; in the disk a rough rule is $A_V \approx 1$ mag per kpc in the $V$ band. The transmitted fraction is $10^{-A_\lambda/2.5}$. We use the standard **Cardelli, Clayton & Mathis (1989)** extinction curve with $R_V = 3.1$.
2. **Atmospheric transmission $T_{\rm atm}(\lambda)$.** For a ground-based telescope, Rayleigh scattering, aerosols and molecular absorption remove light (more in the blue). We use a simple analytic model for a good site at zenith.
3. **Optical throughput $\eta$.** Every mirror and lens reflects/transmits less than 100 %; a real system delivers maybe $\eta\sim0.5$ of the light to the detector.
4. **Detector quantum efficiency $\mathrm{QE}(\lambda)$.** A CCD converts only a fraction of arriving photons into measured electrons, peaking around 80–90 % in the red and falling off in the blue and NIR.

The flux *delivered to the detector* is

$$ f_\lambda^{\rm det} = f_\lambda^{\rm ideal}\;\underbrace{10^{-A_\lambda/2.5}}_{\rm dust}\;\underbrace{T_{\rm atm}(\lambda)}_{\rm sky}\;\underbrace{\eta}_{\rm optics}\;\underbrace{\mathrm{QE}(\lambda)}_{\rm detector}. $$

In [ ]:
from dust_extinction.parameter_averages import CCM89

# --- YOUR CODE HERE ---
# 1. dust:    ext_model = CCM89(Rv=3.1); trans_dust = ext_model.extinguish(lam_grid, Av=1.0)
# 2. atmosphere: write a toy transmission function of wavelength (more loss in the blue)
# 3. optics:  pick a grey throughput eta ~ 0.5
# 4. detector: write a toy QE(lambda) peaking around 650 nm
# 5. multiply them together, apply to f_lam_ideal, and plot ideal vs realistic flux


### 4.3 A familiar yardstick: the apparent magnitude

Astronomers compress "how bright does it look" into the **apparent magnitude** $m$ (a logarithmic, *backwards* scale: smaller = brighter). It relates to the **absolute magnitude** $M$ (the brightness the star *would* have at the reference distance of 10 pc) through the **distance modulus**:

$$ m = M + 5\log_{10}\!\left(\frac{d}{10\,\mathrm{pc}}\right) + A . $$

For the Sun in the $V$ band, $M_V^\odot = 4.83$. At $d=1$ kpc the distance modulus is $5\log_{10}(1000/10) = 5\log_{10}(100) = 10$ mag.
- **Ideal** (no dust): $m_V = 4.83 + 10 = 14.83$.
- **Realistic** (with $A_V=1$): $m_V = 4.83 + 10 + 1 = 15.83$.

A 10 m-class telescope reaches well past $m_V\sim27$, so the Sun at 1 kpc would be an **easy, bright target** — but far too faint to see with the naked eye (limit $m_V\approx6$).

In [ ]:
# --- YOUR CODE HERE ---
# Use m = M + 5 log10(d/10pc) + A, with M_V_sun = 4.83, to get the ideal and
# realistic apparent magnitudes of the Sun at 1 kpc.


---

## 5. Question 1.d — Counting photons in a narrow band over one hour

Light is **quantised**: it arrives in photons, each carrying energy

$$ E_\gamma = \frac{hc}{\lambda}. $$

To turn a power (W = J/s) into a **photon rate** (photons/s) we divide by the energy per photon, then multiply by the exposure time. We work in a **narrow band** of width $\Delta\lambda$ centred on $\lambda_0 = 550$ nm (green / $V$ band), so $L_\lambda$, $f_\lambda$ and $E_\gamma$ are all approximately constant across the band, and the integral $\int L_\lambda\,d\lambda$ is just $L_\lambda(\lambda_0)\,\Delta\lambda$.

We take $\Delta\lambda = 10$ nm and an exposure $t = 1$ hour $= 3600$ s.

| Quantity | Formula |
|----------|---------|
| Photon energy | $E_\gamma = hc/\lambda_0$ |
| **Emitted** by the star | $N_{\rm emit} = \dfrac{L_\lambda(\lambda_0)\,\Delta\lambda}{E_\gamma}\,t$ |
| **Received (ideal)** at the aperture | $N_{\rm rec}^{\rm ideal} = \dfrac{f_\lambda^{\rm ideal}(\lambda_0)\,A_{\rm tel}\,\Delta\lambda}{E_\gamma}\,t$ |
| **Detected (realistic)** | $N_{\rm det}^{\rm real} = N_{\rm rec}^{\rm ideal}\times(\text{dust}\cdot T_{\rm atm}\cdot\eta\cdot\mathrm{QE})$ |

In [ ]:
# --- YOUR CODE HERE ---
# lam0 = 550 nm, dlam = 10 nm, t_exp = 1 hour
# 1. photon energy  E_gamma = h c / lam0
# 2. STEP 1 (ideal): N_emit and N_rec_ideal using L_lambda and f_lambda at lam0
# 3. STEP 2 (real):  multiply N_rec_ideal by the combined loss factor at 550 nm
# 4. print all three numbers and the detected/emitted fraction


---

## 6. Question 2 — The relation between emitted light, distance and received light

Collecting everything above, the number of photons we **detect** in a fixed band and exposure is

$$ N_{\rm det} \;=\; \underbrace{\frac{L_\lambda\,\Delta\lambda}{E_\gamma}\,t}_{\text{emitted rate}}\;\times\;\underbrace{\frac{A_{\rm tel}}{4\pi d^2}}_{\text{geometric capture}}\;\times\;\underbrace{(\text{loss factors})}_{\le 1}. $$

Two clean scaling laws fall out, holding everything else fixed:

$$ \boxed{\,N_{\rm det} \propto \dfrac{L}{d^{2}}\,}\qquad\text{and}\qquad \boxed{\,N_{\rm det}\propto A_{\rm tel}\propto D^{2}\,}. $$

- **Distance:** the inverse-square law. Move a source 10× farther and it delivers $100\times$ fewer photons. On a log–log plot, $N$ vs $d$ is a straight line of **slope $-2$**.
- **Telescope:** photon count grows as the **collecting area**, i.e. as the **square of the diameter**. A 10 m mirror gathers $(10/1)^2 = 100\times$ more light than a 1 m mirror. On a log–log plot, $N$ vs $D$ has **slope $+2$**.

The realistic curve is the ideal curve scaled down by the (here roughly constant) loss factor — a *parallel* line on a log scale, not a change of slope.

### 6.1 Question 2.a — Vary the distance and the telescope diameter

In [ ]:
# --- YOUR CODE HERE ---
# 1. write detected_photons(d, D) returning photons/hour in the 550 nm band
#    (optionally with/without the loss factor)
# 2. (a) hold D = 10 m, sweep distance 1 pc -> 10 kpc; plot N vs d on log-log axes
# 3. (b) hold d = 1 kpc, sweep diameter 0.1 -> ~40 m; plot N vs D on log-log axes
# 4. fit the log-log slopes and check they are -2 and +2


In [ ]:
# --- YOUR CODE HERE (optional) ---
# Make a 2-D contour map of log10(detected photons/hour) over a grid of
# distance (y) and telescope diameter (x); overlay the detection-floor contour
# and mark the (10 m, 1 kpc) operating point.


---

## 7. Summary of results

The cell below gathers every number computed above into one table.

In [ ]:
# --- YOUR CODE HERE ---
# Print a tidy summary table of all the quantities you computed.


---

## 8. Exercises — your turn

Work these in fresh cells. They reuse the functions you have already written.

1. **A cooler star.** Replace the Sun with a red M-dwarf: $T = 3200$ K, $R = 0.3\,R_\odot$. Where does its spectrum peak (Wien)? Recompute its luminosity (Stefan–Boltzmann) and the photons detected at 1 kpc with the 10 m telescope, *in the same 550 nm band*. Is it easier or harder to detect than the Sun, and why? (Hint: think about where an M-dwarf emits most of its light.)

2. **Go to space.** A space telescope sees **no atmosphere and no extra optics losses you can avoid**, but interstellar dust is still there. Recompute $N_{\rm det}$ for the Sun at 1 kpc dropping only the atmospheric term. By what factor does the photon count improve?

3. **How far can we see it?** Using `detected_photons`, find the distance at which the Sun's detected photon count in 1 hour drops to the detection floor of 100 photons, for $D = 1$ m, $10$ m and $39$ m (the ELT). Comment on how the reach scales with $D$.

4. **Widen the band.** Redo Question 1.d with $\Delta\lambda = 100$ nm instead of 10 nm. Does the photon count scale exactly linearly with $\Delta\lambda$? When would the narrow-band approximation ($L_\lambda \approx$ const across the band) start to break down?

---

### Key formulae to remember

| Concept | Formula |
|---------|---------|
| Planck radiance | $B_\lambda = \dfrac{2hc^2}{\lambda^5}\dfrac{1}{e^{hc/\lambda k_BT}-1}$ |
| Stefan–Boltzmann | $L = 4\pi R^2\sigma T^4$ |
| Wien's law | $\lambda_{\rm peak} = b/T$ |
| Spectral luminosity | $L_\lambda = 4\pi^2 R^2 B_\lambda$ |
| Inverse-square flux | $f_\lambda = L_\lambda/(4\pi d^2)$ |
| Collecting area | $A = \pi (D/2)^2$ |
| Photon energy | $E_\gamma = hc/\lambda$ |
| Photon count | $N = \dfrac{f_\lambda A\,\Delta\lambda}{E_\gamma}\,t\times(\text{losses})$ |
| Distance modulus | $m = M + 5\log_{10}(d/10\,\mathrm{pc}) + A$ |

**Take-away:** the brightness of a star at your telescope is set by a tug-of-war between how much it *emits* ($L$), how *far* it is ($1/d^2$), and how *big* your mirror is ($D^2$) — and a real measurement then pays a tax (dust, sky, optics, detector) on top.